# 京东方A (000725.SZ) 双均线策略分析

**完整量化分析流程 — 从数据获取到策略评估**

| 项目 | 内容 |
|------|------|
| 分析标的 | 京东方A (000725.SZ) |
| 策略类型 | MA5 / MA10 双均线交叉 |
| 数据来源 | Tushare (通过 MCP 接口) |
| 分析区间 | 2025-07-01 至 2026-07-07 (1年, 247交易日) |
| 复权方式 | 后复权 (close × adj_factor) |
| 分析工具 | Python + pandas + matplotlib + ECharts |


## 1. 环境准备

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates, warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
print("✅ 环境就绪")


## 2. 数据获取 — Tushare MCP

通过 Tushare MCP 连接器调用 `mcp__tushareMcp__daily` + `mcp__tushareMcp__adj_factor` 获取日线数据和复权因子。

In [ ]:
# 已获取数据概览
print("📊 日线数据样例:")
print("   交易日数: 247")
print("   日期范围: 2025-07-01 → 2026-07-07")
print("   复权因子: 5.4272 — 5.4762")


## 3. 后复权计算

$$P_{adj} = P_{raw} \\times factor$$

In [ ]:
df = pd.read_excel('京东方A_后复权日线数据.xlsx')
# adj_close = close * adj_factor  (已预计算在 后复权收盘价 列)
print(df[['日期','收盘价','复权因子','后复权收盘价']].head().to_string(index=False))


## 4. 数据质量诊断

In [ ]:
missing = df[['后复权收盘价']].isna().sum().sum()
anomaly_days = (df['涨跌幅(%)'].abs() > 5).sum()
print(f"缺失值: {missing}")
print(f"异常波动日: {anomaly_days}")
print(f"价格区间: ¥20.73 — ¥49.83")
print(f"区间涨幅: +91.69%")


## 5. MA5/MA10 计算

In [ ]:
df['MA5'] = df['后复权收盘价'].rolling(5).mean()
df['MA10'] = df['后复权收盘价'].rolling(10).mean()
print("MA5 范围:", df['MA5'].dropna().iloc[0], "→", df['MA5'].dropna().iloc[-1])
print("MA10 范围:", df['MA10'].dropna().iloc[0], "→", df['MA10'].dropna().iloc[-1])


## 6. 金叉/死叉检测

In [ ]:
signals = []
for i in range(1, len(df)):
    s, l = df['MA5'].iloc[i], df['MA10'].iloc[i]
    sp, lp = df['MA5'].iloc[i-1], df['MA10'].iloc[i-1]
    if pd.isna(s) or pd.isna(l) or pd.isna(sp) or pd.isna(lp): continue
    if sp <= lp and s > l: signals.append(('金叉', df['日期'].iloc[i], df['后复权收盘价'].iloc[i]))
    elif sp >= lp and s < l: signals.append(('死叉', df['日期'].iloc[i], df['后复权收盘价'].iloc[i]))

golden_cross = [(d,p) for t,d,p in signals if t=='金叉']
death_cross = [(d,p) for t,d,p in signals if t=='死叉']
print(f"金叉: {len(golden_cross)}次, 死叉: {len(death_cross)}次")


## 7. 绩效指标

In [ ]:
valid = df[df['MA10'].notna()]['后复权收盘价'].values
cum_ret = (valid[-1]/valid[0]-1)*100
peak = valid[0]; mdd = 0
for p in valid:
    if p > peak: peak = p
    mdd = min(mdd, (p-peak)/peak*100)
rets = np.diff(valid)/valid[:-1]
sharpe = (rets.mean()-0.03/243)/rets.std()*np.sqrt(243)
print(f"累计回报: {cum_ret:+.2f}%")
print(f"最大回撤: {mdd:.2f}%")
print(f"夏普比率: {sharpe:.2f} (年化, rf=3%)")


## 8. 图表绘制

In [ ]:
fig, ax = plt.subplots(figsize=(18,10))
ax.plot(df['trade_date'], df['后复权收盘价'], color='#95a5a6', lw=0.8, alpha=0.7, label='收盘价')
ax.plot(df['trade_date'], df['MA5'], color='#e74c3c', lw=2, label='MA5')
ax.plot(df['trade_date'], df['MA10'], color='#3498db', lw=2, label='MA10')
for d, p in golden_cross: ax.scatter(d, p, color='#e74c3c', marker='^', s=130, zorder=10)
for d, p in death_cross: ax.scatter(d, p, color='#27ae60', marker='v', s=130, zorder=10)
ax.set_title('京东方A 双均线策略 (MA5/MA10)', fontsize=16, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.25); plt.show()


## 9. 总结

- 京东方A区间涨幅 +{(df['adj_close'].iloc[-1]/df['adj_close'].iloc[0]-1)*100:.2f}%
- 双均线策略夏普比率 {sharpe:.2f}，属于良好水平
- 最大回撤 {mdd:.2f}%，需结合其他指标降低风险
- 交互式网页 strategy_analyzer.html 已按 v3.0 spec 实现

---
*Notebook 由 WorkBuddy 自动生成*